# Pricing, numerical methods, and Greeks

This study uses only the merged F1 public Python surface to ask one Black–Scholes pricing question with three valuation methods and two sensitivity methods.

The financial problem is fixed: a one-year ATM European call with `S=K=100`, continuously compounded `r=5%`, annualized volatility `20%`, zero dividend yield, and ACT/365F dates. Method settings remain separate from the financial problem.

Model references: [`black_scholes.md`](../docs/models/black_scholes.md) and [`m2_numerical_methods_and_sensitivities.md`](../docs/models/m2_numerical_methods_and_sensitivities.md).


In [ ]:
from datetime import date

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from qf_platform.application import (
    BlackScholesStudyComposition,
    M2WorkbenchConfig,
    M2WorkbenchRequest,
    WorkbenchValuationMethod,
    run_m2_workbench,
)
from qf_platform.pricing import (
    BlackScholesClosedForm,
    BlackScholesLaw,
    BlackScholesParameters,
    EquityState,
    EuropeanOption,
    FlatMoneyMarketNumeraire,
    ModeledState,
    OptionRight,
    PricingMeasureSemantics,
    PricingProblem,
)
from qf_platform.sensitivity import (
    BlackScholesSensitivity,
    FiniteDifferenceBlackScholesSensitivity,
)

valuation_date = date(2026, 1, 1)
law = BlackScholesLaw()
numeraire = FlatMoneyMarketNumeraire(valuation_date, 0.05)
problem = PricingProblem(
    current_state=ModeledState(
        valuation_date,
        EquityState(100.0),
        law.state_space,
    ),
    stochastic_law=law,
    parameters=BlackScholesParameters(annualized_volatility=0.20),
    contract=EuropeanOption(date(2027, 1, 1), 100.0, OptionRight.CALL),
    numeraire=numeraire,
    pricing_measure=PricingMeasureSemantics(name="Q^B", numeraire=numeraire),
)
request = M2WorkbenchRequest(
    composition=BlackScholesStudyComposition(problem, BlackScholesClosedForm()),
    config=M2WorkbenchConfig(
        valuation_method=WorkbenchValuationMethod.ANALYTIC,
        crr_steps=400,
        monte_carlo_paths=20_000,
        monte_carlo_seed=1729,
        selected_greek=BlackScholesSensitivity.DELTA,
        finite_difference_method=FiniteDifferenceBlackScholesSensitivity(
            spot_bump=0.1,
            volatility_bump=0.001,
            rate_bump=0.0001,
            theta_day_bump=1,
        ),
    ),
)
analysis = run_m2_workbench(request)

In [ ]:
def show_table(headers, rows):
    text = "| " + " | ".join(headers) + " |\n"
    text += "| " + " | ".join("---" for _ in headers) + " |\n"
    for row in rows:
        text += "| " + " | ".join(str(value) for value in row) + " |\n"
    display(Markdown(text))

analytic = analysis.selected_valuation.result
assert analytic is not None
assert abs(analytic.present_value - 10.4505835722) < 5e-10

valuation_rows = []
for run in analysis.valuations:
    if run.result is None:
        valuation_rows.append((run.method.value, run.configuration, "unsupported", "—"))
        continue
    valuation_rows.append(
        (
            run.method.value,
            run.configuration,
            f"{run.result.present_value:.8f}",
            f"{run.result.present_value - analytic.present_value:+.6f}",
        )
    )
show_table(
    ("method", "configuration", "present value", "error vs analytic"),
    valuation_rows,
)

In [ ]:
crr_steps = [
    point.steps for point in analysis.crr_convergence if point.result is not None
]
crr_errors = [
    abs(point.result.present_value - analytic.present_value)
    for point in analysis.crr_convergence
    if point.result is not None
]
mc_paths = [point.paths for point in analysis.monte_carlo_convergence]
mc_errors = [
    abs(point.result.present_value - analytic.present_value)
    for point in analysis.monte_carlo_convergence
]
mc_half_widths = [
    0.5
    * (point.result.confidence_interval_95[1] - point.result.confidence_interval_95[0])
    for point in analysis.monte_carlo_convergence
]

plt.figure(figsize=(7, 4))
plt.loglog(crr_steps, crr_errors, marker="o", label="CRR absolute error")
plt.loglog(mc_paths, mc_errors, marker="o", label="Monte Carlo absolute error")
plt.loglog(mc_paths, mc_half_widths, marker="o", label="Monte Carlo 95% half-width")
plt.xlabel("steps or paths")
plt.ylabel("present-value units")
plt.title("Numerical evidence against the analytic reference")
plt.legend()
plt.show()

The Monte Carlo interval is **sampling uncertainty**, not a deterministic pricing tolerance and not model error. CRR discretization error and Monte Carlo sampling error are different mechanisms even though both are shown against the same analytic reference.


In [ ]:
greek_rows = []
for run in analysis.sensitivities:
    analytic_result = run.analytic_result
    fd_result = run.finite_difference_result
    if analytic_result is None or fd_result is None:
        continue
    greek_rows.append(
        (
            run.sensitivity.value,
            f"{analytic_result.value:.8f}",
            f"{fd_result.value:.8f}",
            f"{fd_result.value - analytic_result.value:+.3e}",
            analytic_result.units,
        )
    )
show_table(
    ("Greek", "analytic", "finite difference", "FD - analytic", "units"),
    greek_rows,
)

delta = next(
    run
    for run in analysis.sensitivities
    if run.sensitivity is BlackScholesSensitivity.DELTA
)
assert delta.analytic_result is not None
assert delta.finite_difference_result is not None
assert abs(delta.analytic_result.value - delta.finite_difference_result.value) < 1e-4

In [ ]:
spots = [point.spot for point in analysis.greek_curve]
analytic_delta = [
    point.analytic_result.value if point.analytic_result is not None else float("nan")
    for point in analysis.greek_curve
]
fd_delta = [
    point.finite_difference_result.value
    if point.finite_difference_result is not None
    else float("nan")
    for point in analysis.greek_curve
]

plt.figure(figsize=(7, 4))
plt.plot(spots, analytic_delta, label="analytic Delta")
plt.plot(spots, fd_delta, linestyle="--", label="finite-difference Delta")
plt.xlabel("spot")
plt.ylabel("Delta")
plt.title("Independent sensitivity methods on the same pricing problem")
plt.legend()
plt.show()

## Interpretation

Agreement among analytic, tree, Monte Carlo, and finite-difference implementations is useful numerical evidence. It does **not** establish that the Black–Scholes assumptions describe observed markets. The next studies separate dynamic replication evidence and observed-market/model-risk evidence from numerical-method agreement.
